# Flight Delay Prediction - Feature Engineering

## Objective
This notebook focuses on transforming raw flight data into meaningful features
that improve model performance.

We will:
- Define the target variable
- Extract time-based features
- Remove data leakage features
- Select relevant columns
- Encode categorical variables
- Prepare final dataset for modeling


### Step 1 : Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10,6)

### Step 2 : Load the Dataset

In [4]:
df = pd.read_csv("/content/Flight_delay.csv")

In [5]:
df.head()

,DayOfWeek,Date,DepTime,ArrTime,CRSArrTime,UniqueCarrier,Airline,FlightNum,TailNum,ActualElapsedTime,...,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,4,03-01-2019,1829,1959,1925,WN,Southwest Airlines Co.,3920,N464WN,90,...,3,10,0,N,0,2,0,0,0,32
1,4,03-01-2019,1937,2037,1940,WN,Southwest Airlines Co.,509,N763SW,240,...,3,7,0,N,0,10,0,0,0,47
2,4,03-01-2019,1644,1845,1725,WN,Southwest Airlines Co.,1333,N334SW,121,...,6,8,0,N,0,8,0,0,0,72
3,4,03-01-2019,1452,1640,1625,WN,Southwest Airlines Co.,675,N286WN,228,...,7,8,0,N,0,3,0,0,0,12
4,4,03-01-2019,1323,1526,1510,WN,Southwest Airlines Co.,4,N674AA,123,...,4,9,0,N,0,0,0,0,0,16


In [6]:
df.shape

(484551, 29)

### Step 3 : Crate target variable

In [7]:
df["is_delayed"] = (df["ArrDelay"] > 15).astype(int)

In [8]:
df["is_delayed"].value_counts()

,count
is_delayed,
1,471530
0,13021


### Target Definition

A flight is considered delayed if arrival delay exceeds 15 minutes.
This threshold aligns with aviation industry standards.

This target variable will be used for classification.


### Step 4 : Remove Data Leakage Columns

In [9]:
leakage_columns = [
    "ArrDelay",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
]

In [10]:
df = df.drop(columns=leakage_columns)

In [11]:
df.head()

,DayOfWeek,Date,DepTime,ArrTime,CRSArrTime,UniqueCarrier,Airline,FlightNum,TailNum,ActualElapsedTime,...,Org_Airport,Dest,Dest_Airport,Distance,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,is_delayed
0,4,03-01-2019,1829,1959,1925,WN,Southwest Airlines Co.,3920,N464WN,90,...,Indianapolis International Airport,BWI,Baltimore-Washington International Airport,515,3,10,0,N,0,1
1,4,03-01-2019,1937,2037,1940,WN,Southwest Airlines Co.,509,N763SW,240,...,Indianapolis International Airport,LAS,McCarran International Airport,1591,3,7,0,N,0,1
2,4,03-01-2019,1644,1845,1725,WN,Southwest Airlines Co.,1333,N334SW,121,...,Indianapolis International Airport,MCO,Orlando International Airport,828,6,8,0,N,0,1
3,4,03-01-2019,1452,1640,1625,WN,Southwest Airlines Co.,675,N286WN,228,...,Indianapolis International Airport,PHX,Phoenix Sky Harbor International Airport,1489,7,8,0,N,0,0
4,4,03-01-2019,1323,1526,1510,WN,Southwest Airlines Co.,4,N674AA,123,...,Indianapolis International Airport,TPA,Tampa International Airport,838,4,9,0,N,0,1


### Data Leakage Prevention

Delay-related columns such as ArrDelay and delay reason columns were removed.
These values are known only after flight completion and would leak future information into the model.


### Step 4 : Time based Features Extraction

In [12]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

In [13]:
df["Month"] = df["Date"].dt.month
df["DayOfMonth"] = df["Date"].dt.day

df["Hour"] = (df["DepTime"] // 100).astype(int)

In [14]:
df[["Month", "DayOfMonth", "Hour"]].head()

,Month,DayOfMonth,Hour
0,3.0,1.0,18
1,3.0,1.0,19
2,3.0,1.0,16
3,3.0,1.0,14
4,3.0,1.0,13


### Time-Based Features

- Month: Captures seasonal effects
- DayOfMonth: Captures monthly patterns
- Hour: Captures peak traffic and congestion hours


### Step 6 : Drop Irrelavant columns

In [18]:
selected_features = [
    "Airline",
    "Origin",
    "Dest",
    "Month",
    "DayOfMonth",
    "DayOfWeek",
    "Hour",
    "Distance",
    "is_delayed"   # keep target
]

In [19]:
df = df[selected_features]

In [20]:
df.head()

,Airline,Origin,Dest,Month,DayOfMonth,DayOfWeek,Hour,Distance,is_delayed
0,Southwest Airlines Co.,IND,BWI,3.0,1.0,4,18,515,1
1,Southwest Airlines Co.,IND,LAS,3.0,1.0,4,19,1591,1
2,Southwest Airlines Co.,IND,MCO,3.0,1.0,4,16,828,1
3,Southwest Airlines Co.,IND,PHX,3.0,1.0,4,14,1489,0
4,Southwest Airlines Co.,IND,TPA,3.0,1.0,4,13,838,1


### Final Feature Selection

After removing data leakage and irrelevant identifiers,
only the following features were retained for modeling:

- Airline
- Origin
- Destination
- Month
- DayOfMonth
- DayOfWeek
- Hour
- Distance

These features are available before flight departure
and are realistic inputs for prediction.

All other columns were removed to:
- Avoid data leakage
- Improve model generalization
- Simplify deployment
- Ensure consistency between training and prediction layers


### Step 7 : Encode Categorical Variables

In [21]:
categorical_cols = df.select_dtypes(include="object").columns

In [22]:
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [23]:
df_encoded.head()

,Month,DayOfMonth,DayOfWeek,Hour,Distance,is_delayed,Airline_American Airlines Inc.,Airline_American Eagle Airlines Inc.,Airline_Atlantic Southeast Airlines,Airline_Delta Air Lines Inc.,...,Dest_TYR,Dest_TYS,Dest_VLD,Dest_VPS,Dest_WRG,Dest_WYS,Dest_XNA,Dest_YAK,Dest_YKM,Dest_YUM
0,3.0,1.0,4,18,515,1,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,3.0,1.0,4,19,1591,1,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,3.0,1.0,4,16,828,1,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,3.0,1.0,4,14,1489,0,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,3.0,1.0,4,13,838,1,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


Categorical variables were encoded using one-hot encoding
to allow machine learning algorithms to process them numerically.


### Step 8 : Final Dataset Overview

In [24]:
df_encoded.shape

(484551, 563)

In [25]:
df_encoded.head()

,Month,DayOfMonth,DayOfWeek,Hour,Distance,is_delayed,Airline_American Airlines Inc.,Airline_American Eagle Airlines Inc.,Airline_Atlantic Southeast Airlines,Airline_Delta Air Lines Inc.,...,Dest_TYR,Dest_TYS,Dest_VLD,Dest_VPS,Dest_WRG,Dest_WYS,Dest_XNA,Dest_YAK,Dest_YKM,Dest_YUM
0,3.0,1.0,4,18,515,1,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,3.0,1.0,4,19,1591,1,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,3.0,1.0,4,16,828,1,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,3.0,1.0,4,14,1489,0,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,3.0,1.0,4,13,838,1,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


### Final Dataset Summary

After feature engineering:
- Data leakage has been removed
- Time-based features have been added
- Categorical variables have been encoded
- The dataset is now ready for model training
